# Lab 10: Convolutional neural networks

In this lab, we will build convolutional neural networks to make predictions. Let us start with our well-known handwritten digits dataset. We begin by loading and scaling the data, and splitting them into a training and a testing set. We can reuse our code from the previous chapter.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

# ==== PARAMETERS ====
batch_size = 64
epochs = 50
learning_rate = 1e-3
device = "cuda" if torch.cuda.is_available() else "cpu"

# ==== LOAD DATA ====
digits = load_digits()
X = digits.data  # shape (1797, 64)
y = digits.target

# Scale features
X = MinMaxScaler().fit_transform(X)

# Reshape into (N, 1, 8, 8)
X = X.reshape(-1, 1, 8, 8)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

# DataLoaders
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

If you look at the code carefully, there is one **important** difference between the above code and the code from the previous chapter: the data was reshaped to have shape (1797, 1, 8, 8). This is accomplished by the line:

X = X.reshape(-1, 1, 8, 8)

Having the proper shape is important for constructing CNNs. The <a href="https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html" target="_blank">Conv2d</a> layers are expecting an input of shape (number of samples, number of channels, height, width).

Let us now define a simple CNN to predict the handwritten digits.

In [4]:
# ==== DEFINE MODEL ====
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 2 * 2, 64)
        self.fc2 = nn.Linear(64, 10)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # (1,8,8) -> (16,4,4)
        x = self.pool(self.relu(self.conv2(x)))   # (16,4,4) -> (32,2,2)
        x = x.view(x.size(0), -1)                 # flatten
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)

Let us examine the above code carefully. We use the following layers:

1. A convolution layer with 1 input channel, 16 filters of size $3 \times 3$, and a padding of $1$. 
2. A $2 \times 2$ max pooling layer.
3. A convolution layer with the 16 input channels from the previous convolution layer, 32 filters of size $3 \times 3$, and a padding of $1$. 
4. A linear layer with $64$ outputs and ReLU activation.
5. A linear layer with $10$ outputs and no activation.

As we move through the layers, it is important to keep track of the size of the images. The first convolution takes the original images ($1$ channel) of size $8 \times 8$, and outputs $16$ images of dimension

$$
m + 2 p_h - k + 1 = n + 2 p_w -l + 1 = 8 + 2 \times 1 - 3 + 1 = 8. 
$$

We therefore have an output of size $(16, 8, 8)$. Next, the max pool reduces the dimension to $(16, 4, 4)$. 

The next convolution layer produces images of size

$$
m + 2 p_h - k + 1 = n + 2 p_w -l + 1 = 4 + 2 \times 1 - 3 + 1 = 4. 
$$

The dimension of the output is therefore $(32,4,4)$. Applying the max pool yields an output of size $(32,2,2)$. All this data is flattened and passed to a linear layer. The input dimension of this layer has to be $32*2*2 = 128$. We therefore have a fully connected layer with $128$ inputs and $64$ outputs (with ReLU activation), and then a fully connected layer with $64$ inputs and $10$ outputs (no activation). The final output consists of raw scores for classifying the digits in one of the $10$ categories. In general, in a classification problem with $K$ categories, one would have $K$ outputs in the last layer. 

It may be tempting to apply a sigmoid activation in the last layer, and maybe to even rescale the output to obtain a probability distribution on the digits $0, 1, \dots, 9$. However, the <a href="https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html" target="_blank">CrossEntropyLoss</a> loss function that we will use below does that automatically for us. It expects raw scores as input. 

```{warning} 
The <a href="https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html" target="_blank">CrossEntropyLoss</a> loss function expects raw scores which do not need to be positive or sum to 1. These scores are then internally converted to properly evaluate cross-entropy.

```

The rest of the code is similar to our previous code, except we will now use the CrossEntropyLoss function. We conclude by evaluating the performance of the trained model on the testing set.

In [7]:
# ==== TRAINING SETUP ====
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# ==== TRAIN LOOP ====
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Evaluate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {total_loss/len(train_loader):.4f} - Test Acc: {acc:.2f}%")

# ==== FINAL ACCURACY ====
print("Final test accuracy:", acc)

Epoch [1/50] - Loss: 0.1009 - Test Acc: 96.11%
Epoch [2/50] - Loss: 0.0897 - Test Acc: 96.67%
Epoch [3/50] - Loss: 0.0787 - Test Acc: 98.06%
Epoch [4/50] - Loss: 0.0722 - Test Acc: 98.06%
Epoch [5/50] - Loss: 0.0648 - Test Acc: 97.22%
Epoch [6/50] - Loss: 0.0691 - Test Acc: 96.67%
Epoch [7/50] - Loss: 0.0692 - Test Acc: 98.06%
Epoch [8/50] - Loss: 0.0614 - Test Acc: 96.94%
Epoch [9/50] - Loss: 0.0662 - Test Acc: 97.78%
Epoch [10/50] - Loss: 0.0519 - Test Acc: 97.22%
Epoch [11/50] - Loss: 0.0439 - Test Acc: 96.67%
Epoch [12/50] - Loss: 0.0500 - Test Acc: 98.06%
Epoch [13/50] - Loss: 0.0426 - Test Acc: 97.50%
Epoch [14/50] - Loss: 0.0404 - Test Acc: 98.33%
Epoch [15/50] - Loss: 0.0343 - Test Acc: 98.33%
Epoch [16/50] - Loss: 0.0309 - Test Acc: 98.06%
Epoch [17/50] - Loss: 0.0299 - Test Acc: 98.06%
Epoch [18/50] - Loss: 0.0276 - Test Acc: 97.22%
Epoch [19/50] - Loss: 0.0289 - Test Acc: 97.50%
Epoch [20/50] - Loss: 0.0310 - Test Acc: 98.89%
Epoch [21/50] - Loss: 0.0254 - Test Acc: 98.06%
E

After only a few seconds of training, we obtain an excellent test accuracy.

## Summary

Some key takeaways from the above exercise: 

1. Before applying convolutions, it is important to reshape the input using X = X.reshape(-1, nb_channels, height, width).
2. In order to feed the convolution layers into a fully connected layer, one needs to keep track of the size of the outputs. 
3. In classification problems, the number of outputs of the last layer needs to corresponds to the number of categories/labels of the data. 
4. In classification problems, the CrossEntropyLoss function expects raw scores; no normalization or activation is needed in the last layer.

## Exercises

(a) Try modifying the above code on your own. Can you do better by 

1. Changing the layer sizes, number of filters, number of layers, etc.? 
2. Using dropouts? 

(b) The Fashion-MNIST dataset contains 70,000 labeled grayscale images of clothes.

```{figure} images/fashion-mnist.png
---
width: 500 px
---
Sample of images from the Fashion-MNIST dataset.
```

Each image has a label 0 to 9 corresponding to: 

T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot.


Load the Fashion-MNIST dataset and construct a good CNN model to predict the labels. You can use the code below to load the dataset. 

In [9]:
'''
# --- Example of code to load the Fashion-MNIST dataset

import torch
from torchvision import datasets, transforms

# Transform: normalize to [0,1] and convert to tensor
transform = transforms.Compose([
    transforms.ToTensor()
])

# Load dataset
train_data = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform)

# Data loaders
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=64, shuffle=False)

# Access an example
images, labels = next(iter(train_loader))
print(images.shape)  # torch.Size([64, 1, 28, 28])
print(labels[:10])

'''

"\n# --- Example of code to load the Fashion-MNIST dataset\n\nimport torch\nfrom torchvision import datasets, transforms\n\n# Transform: normalize to [0,1] and convert to tensor\ntransform = transforms.Compose([\n    transforms.ToTensor()\n])\n\n# Load dataset\ntrain_data = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform)\ntest_data = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform)\n\n# Data loaders\ntrain_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)\ntest_loader = torch.utils.data.DataLoader(test_data, batch_size=64, shuffle=False)\n\n# Access an example\nimages, labels = next(iter(train_loader))\nprint(images.shape)  # torch.Size([64, 1, 28, 28])\nprint(labels[:10])\n\n"